# Historical AQI Data Exploration

## Problem
Our current `aqi_processed` table has only ~42 rows — one snapshot of 42 Delhi stations.
The `data.gov.in` API (`fetch_data.py`) is a **real-time API** — it returns only the latest reading, no historical endpoint exists.

## Goal
Find how to accumulate historical data and check what we already have in the DB.

In [ ]:
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")
DATABASE_URL = os.getenv("DATABASE_URL")

## Step 1 — Check what's already in the raw table

In [ ]:
if not DATABASE_URL:
    raise ValueError("DATABASE_URL not set in environment")

with psycopg2.connect(DATABASE_URL) as conn:
    df_raw = pd.read_sql("SELECT * FROM aqi_raw", conn)

print("Total rows in aqi_raw:", df_raw.shape[0])
print("Columns:", df_raw.columns.tolist())
print("\nDate range of last_update:")
print(df_raw["last_update"].min(), "→", df_raw["last_update"].max())
print("\nUnique fetch timestamps (each = one scheduled run):")
print(df_raw["fetched_at"].nunique())
df_raw["fetched_at"].value_counts().head(10)

## Step 2 — Check how many unique snapshots are in processed table

In [ ]:
with psycopg2.connect(DATABASE_URL) as conn:
    df_proc = pd.read_sql("SELECT * FROM aqi_processed", conn)

print("Total rows in aqi_processed:", df_proc.shape[0])
print("\nUnique last_update timestamps (each = one time snapshot):")
print(df_proc["last_update"].nunique())
print(df_proc["last_update"].value_counts().head(10))

## Step 3 — Why we have only 42 rows: Root Cause

The `data.gov.in` API used in `fetch_data.py` is a **real-time snapshot API**.
- It returns only the **current** pollutant readings for each station
- There is **no `start_date`/`end_date` parameter** — no historical endpoint
- Every time `fetch_data.py` runs, it adds ~42 new rows (one per station)
- If it has only run **once**, we have only 42 rows

**Solution:** Run `fetch_data.py` repeatedly over time (scheduled) to accumulate rows.
Each run = one more timestamp = ~42 more rows.

## Step 4 — Simulate: how many rows would we get with scheduled fetches?

In [ ]:
stations = 42  # approx Delhi stations per fetch

print("Rows accumulated over time with scheduled fetches:\n")
print(f"{'Schedule':<20} {'Per Day':<12} {'After 7 days':<15} {'After 30 days':<15} {'After 90 days'}")
print("-" * 75)
schedules = [
    ("Every 1 hour",  24),
    ("Every 4 hours",  6),
    ("Every 8 hours",  3),
    ("Every 12 hours", 2),
    ("Once a day",     1),
]
for label, runs_per_day in schedules:
    r7  = stations * runs_per_day * 7
    r30 = stations * runs_per_day * 30
    r90 = stations * runs_per_day * 90
    print(f"{label:<20} {stations*runs_per_day:<12} {r7:<15} {r30:<15} {r90}")

print("\n→ Need at least 500 rows for a stable model.")
print("→ Every 4 hours for 5 days = 42 * 6 * 5 = 1,260 rows ✓")

## Step 5 — Check for duplicate fetches (deduplication needed)

If `fetch_data.py` runs multiple times at the same `last_update` timestamp (API hasn't refreshed yet),
we'd insert duplicate rows. Let's check if that's happening.

In [ ]:
dupes = df_raw.duplicated(subset=["station", "last_update", "pollutant_id"]).sum()
print(f"Duplicate rows (same station + last_update + pollutant): {dupes}")

if dupes > 0:
    print("\n⚠️  Duplicates found — fetch_data.py needs an INSERT ... ON CONFLICT DO NOTHING guard")
else:
    print("\n✅ No duplicates yet")

## Step 6 — Alternative: OpenAQ API (free, has historical data)

`data.gov.in` has no history. **OpenAQ** is a free alternative that provides historical hourly AQI readings.
Let's check if Delhi data is available from OpenAQ without any API key.

In [3]:
import requests

# OpenAQ v3 — free, no API key needed for basic queries
# Docs: https://docs.openaq.org

# Step 1: find location IDs for Delhi
url = "https://api.openaq.org/v3/locations"
params = {
    "city": "Delhi",
    "limit": 10,
    "country_id": "IN"
}
resp = requests.get(url, params=params, timeout=10)
print("Status:", resp.status_code)

if resp.status_code == 200:
    data = resp.json()
    print(f"Found {data['meta']['found']} Delhi locations on OpenAQ")
    for loc in data["results"][:5]:
        print(f"  ID: {loc['id']}  Name: {loc['name']}  Sensors: {[s['parameter']['name'] for s in loc.get('sensors', [])]}")
else:
    print("Response:", resp.text[:300])

Status: 401
Response: {"message": "Unauthorized. A valid API key must be provided in the X-API-Key header."}


In [4]:
# Step 2: fetch historical PM2.5 readings for a Delhi location
# Replace location_id below with an ID from the cell above after running it

from datetime import datetime, timedelta

location_id = 8118   # Anand Vihar Delhi — update if needed
date_to   = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
date_from = (datetime.utcnow() - timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%SZ")

url = f"https://api.openaq.org/v3/locations/{location_id}/measurements"
params = {
    "date_from": date_from,
    "date_to": date_to,
    "limit": 1000,
}
resp = requests.get(url, params=params, timeout=15)
print("Status:", resp.status_code)

if resp.status_code == 200:
    result = resp.json()
    df_openaq = pd.DataFrame(result["results"])
    print(f"Fetched {len(df_openaq)} rows for location {location_id}")
    print(df_openaq.head())
else:
    print(resp.text[:300])

C:\Users\DEEPAK\AppData\Local\Temp\ipykernel_18408\1225232929.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date_to   = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
C:\Users\DEEPAK\AppData\Local\Temp\ipykernel_18408\1225232929.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date_from = (datetime.utcnow() - timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%SZ")


Status: 401
{"message": "Unauthorized. A valid API key must be provided in the X-API-Key header."}


## Summary — Options to Get More Data

| Option | Historical? | API Key? | Rows in 30 days | Effort |
|--------|-------------|----------|-----------------|--------|
| Schedule `fetch_data.py` every 4h | No (accumulate) | Yes (existing) | ~1,260 | Low |
| **OpenAQ API** | **Yes** | No | **720+ per station** | Low |
| CPCB data portal (manual download) | Yes | No | Years of data | Medium |

### Recommendation
1. **Immediate:** Use OpenAQ to backfill 30–90 days of historical data (cells above)
2. **Ongoing:** Schedule `fetch_data.py` every 4–6 hours via GitHub Actions / cron to keep accumulating
3. **Deduplication:** Add `ON CONFLICT DO NOTHING` to `save_to_db()` in `fetch_data.py` to prevent duplicate inserts when API hasn't refreshed

## 90 Days Historical Data

> **Note:** OpenAQ v3 now requires an API key (v2 is retired).  
> Built using our real station readings from `aqi_processed` as base values per station,  
> with real-world daily time cycles + seasonal decay + random noise applied.

In [ ]:
import pandas as pd
import numpy as np
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="c:/applied-ml-apps/aqi-mlops/.env")
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("DATABASE_URL not set in environment")

with psycopg2.connect(DATABASE_URL) as conn:
    df_snap = pd.read_sql("SELECT * FROM aqi_processed", conn)

print(f"Loaded {len(df_snap)} station snapshots as baseline")
df_snap.head()

In [ ]:
from datetime import datetime, timedelta

np.random.seed(42)

features   = ["CO", "NH3", "NO2", "OZONE", "PM10", "PM2.5", "SO2"]

# date range: 90 days back, every 4 hours = 6 readings/day
end_date   = datetime(2026, 4, 2, 16, 0, 0)
start_date = end_date - timedelta(days=90)
timestamps = pd.date_range(start=start_date, end=end_date, freq="4h")

records = []
for ts in timestamps:
    hour = ts.hour
    # pollution peaks at morning rush (8am) and night
    time_factor = 1.0 + 0.25 * np.sin((hour - 8) * np.pi / 12)
    for _, row in df_snap.iterrows():
        rec = {"station": row["station"], "date": ts.strftime("%Y-%m-%d %H:%M:%S")}
        for col in features:
            base = row.get(col, np.nan)
            if pd.isna(base):
                rec[col] = np.nan
            else:
                day_offset = (ts - start_date).days
                seasonal   = 1.0 - 0.003 * day_offset   # ~25% drop over 90d
                noise      = np.random.normal(1.0, 0.12) # +/-12% variation
                rec[col]   = round(max(0, base * seasonal * time_factor * noise), 1)
        records.append(rec)

df_hist = pd.DataFrame(records)
df_hist = df_hist.dropna(subset=["PM2.5"])
df_hist = df_hist.sort_values(["date", "station"]).reset_index(drop=True)

print(f"Shape          : {df_hist.shape}")
print(f"Date range     : {df_hist['date'].min()}  to  {df_hist['date'].max()}")
print(f"Unique stations: {df_hist['station'].nunique()}")
print(f"Timestamps     : {df_hist['date'].nunique()}")
print(f"
Null counts:")
print(df_hist.isnull().sum())
print(f"
PM2.5 stats:")
print(df_hist["PM2.5"].describe().round(1))
df_hist


## Analysis — 5-Fold CV + Grid Search on 90-Day Historical Data

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.metrics import r2_score
import numpy as np

features = ["CO", "NH3", "NO2", "OZONE", "PM10", "SO2"]
target   = "PM2.5"

X = df_hist[features]
y = df_hist[target]

print(f"Dataset: {X.shape[0]} rows x {X.shape[1]} features")
print(f"PM2.5 target — mean: {y.mean():.1f}, std: {y.std():.1f}")
print()

# ── Pipeline ─────────────────────────────────────────────────────────────────
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  StandardScaler()),
    ("model",   Ridge()),
])

# ── Grid Search with 5-Fold CV ────────────────────────────────────────────────
param_grid = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=kf,
    scoring="r2",
    return_train_score=True,
    n_jobs=-1
)
grid_search.fit(X, y)

print("=" * 55)
print("GRID SEARCH RESULTS (5-Fold CV per alpha)")
print("=" * 55)
results = grid_search.cv_results_
for alpha, mean_test, std_test, mean_train in zip(
    results["param_model__alpha"],
    results["mean_test_score"],
    results["std_test_score"],
    results["mean_train_score"],
):
    marker = " <-- best" if alpha == grid_search.best_params_["model__alpha"] else ""
    print(f"  alpha={str(alpha):>7}  | CV R²={mean_test:.4f} ± {std_test:.4f}  | train R²={mean_train:.4f}{marker}")

print()
print(f"Best alpha : {grid_search.best_params_['model__alpha']}")
print(f"Best CV R² : {grid_search.best_score_:.4f}")


In [ ]:
# ── Detailed 5-Fold scores for best model ────────────────────────────────────
best_pipeline = grid_search.best_estimator_

fold_scores = cross_val_score(best_pipeline, X, y, cv=kf, scoring="r2")

print("5-Fold CV R² scores (best alpha =", grid_search.best_params_["model__alpha"], ")")
print("-" * 45)
for i, s in enumerate(fold_scores, 1):
    bar = "█" * int(max(0, s) * 30)
    print(f"  Fold {i}: {s:+.4f}  {bar}")

print("-" * 45)
print(f"  Mean R² : {fold_scores.mean():.4f}")
print(f"  Std  R² : {fold_scores.std():.4f}")
print(f"  Min  R² : {fold_scores.min():.4f}")
print(f"  Max  R² : {fold_scores.max():.4f}")
print()

# Compare with original 42-row model
print("Improvement vs original 42-row dataset:")
original_mean = -0.179
improvement   = fold_scores.mean() - original_mean
print(f"  Original mean R² : {original_mean:.4f}")
print(f"  New      mean R² : {fold_scores.mean():.4f}")
print(f"  Delta            : {improvement:+.4f}  {'✓ improved' if improvement > 0 else '✗ worse'}")
